In [ ]:
import numpy as np
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.naive_bayes import BernoulliNB, MultinomialNB

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ag-news-classification-dataset' dataset.
Path to dataset files: /kaggle/input/ag-news-classification-dataset


In [ ]:

df_agnews = pd.read_csv("/kaggle/input/ag-news-classification-dataset/train.csv")
df_agnews_test = pd.read_csv("/kaggle/input/ag-news-classification-dataset/test.csv")

In [ ]:
print("AG News Dataset (First 5 Rows):")
df_agnews.head()

AG News Dataset (First 5 Rows):


,Class Index,Title,Description,cleaned_description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",reuters short sellers wall street dwindling ba...
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,reuters private investment firm carlyle group ...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,reuters soaring crude prices plus worries econ...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,reuters authorities halted oil export flows ma...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...",afp tearaway world oil prices toppling records...


In [ ]:
stopwords = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your",
    "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she",
    "her", "hers", "herself", "it", "its", "itself", "they", "them", "their",
    "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these",
    "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has",
    "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if",
    "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about",
    "against", "between", "into", "through", "during", "before", "after", "above", "below",
    "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further",
    "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each",
    "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so",
    "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now"
}

def preprocess_text(text):
    # Removes URLs (0.5 marks)
    text = re.sub(r"http\S+|www\S+", "", text)
    # Removes punctuation and non-alphanumeric characters (0.5 marks)
    text = re.sub(r"'s\b", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    # Converts text to lowercase (0.5 marks)
    text = text.lower()
    # Removes extra whitespace (0.5 marks)
    text = re.sub(r"\s+", " ", text).strip()
    # Removes stopwords (2 marks)
    words = text.split()
    words = [word for word in words if word not in stopwords]
    return " ".join(words)

In [ ]:
# Apply preprocessing to the 'Description' column (1 mark)
df_agnews["cleaned_description"] = df_agnews["Description"].apply(preprocess_text)
df_agnews_test["cleaned_description"] = df_agnews_test["Description"].apply(preprocess_text)
# Print first 5 preprocessed samples (1 mark)
print(df_agnews.head())

   Class Index                                              Title  \
0            3  Wall St. Bears Claw Back Into the Black (Reuters)   
1            3  Carlyle Looks Toward Commercial Aerospace (Reu...   
2            3    Oil and Economy Cloud Stocks' Outlook (Reuters)   
3            3  Iraq Halts Oil Exports from Main Southern Pipe...   
4            3  Oil prices soar to all-time record, posing new...   

                                         Description  \
0  Reuters - Short-sellers, Wall Street's dwindli...   
1  Reuters - Private investment firm Carlyle Grou...   
2  Reuters - Soaring crude prices plus worries\ab...   
3  Reuters - Authorities have halted oil export\f...   
4  AFP - Tearaway world oil prices, toppling reco...   

                                 cleaned_description  
0  reuters short sellers wall street dwindling ba...  
1  reuters private investment firm carlyle group ...  
2  reuters soaring crude prices plus worries econ...  
3  reuters authorities halte

In [ ]:
class BagofWords:
    def __init__(self):
        self.vocab = {}
    def fit(self, sentences):
        word_freq = {}
        for sentence in sentences:
            words = sentence.split()
            for word in words:
                if word in word_freq:
                    word_freq[word] += 1
                else:
                    word_freq[word] = 1
        sorted_words = sorted(
            word_freq.items(),
            key=lambda x: x[1],
            reverse=True
        )
        top_words = sorted_words
        self.vocab = {}
        index = 0
        for word, _ in top_words:
            self.vocab[word] = index
            index += 1
    def vectorize(self, sentence):
        vec = {}
        for word in sentence.lower().split():
            if word in self.vocab:
                idx = self.vocab[word]
                vec[idx] = vec.get(idx, 0) + 1
        return vec


In [ ]:
X_train_texts = df_agnews["cleaned_description"].values
y_train_m = df_agnews["Class Index"].values
X_test_texts = df_agnews_test["cleaned_description"].values
y_test_m = df_agnews_test["Class Index"].values

In [ ]:
bow = BagofWords()
bow.fit(X_train_texts)
X_train_m = [bow.vectorize(text) for text in X_train_texts]
X_test_m  = [bow.vectorize(text) for text in X_test_texts]


In [ ]:
print(len(X_train_m))
print(len(X_test_m))

120000
7600


In [ ]:
vocab_size = len(bow.vocab)
def dicts_to_array(dict_list, vocab_size, dtype=np.uint8):
    arr = np.zeros((len(dict_list), vocab_size), dtype=dtype)
    for i, d in enumerate(dict_list):
        for idx, count in d.items():
            arr[i, idx] = min(count, 255)
    return arr

X_train_array = dicts_to_array(X_train_m, vocab_size, dtype=np.uint8)
X_test_array  = dicts_to_array(X_test_m, vocab_size, dtype=np.uint8)



In [ ]:
mnb_sklearn = MultinomialNB()
classes = np.unique(y_train_m)

batch_size = 10000
for i in range(0, len(X_train_array), batch_size):
    X_batch = X_train_array[i:i+batch_size]
    y_batch = y_train_m[i:i+batch_size]
    mnb_sklearn.partial_fit(X_batch, y_batch, classes=classes)

y_pred_mnb = mnb_sklearn.predict(X_test_array)

accuracy = accuracy_score(y_test_m, y_pred_mnb)
precision = precision_score(y_test_m, y_pred_mnb, average='weighted')
recall = recall_score(y_test_m, y_pred_mnb, average='weighted')
f1 = f1_score(y_test_m, y_pred_mnb, average='weighted')
cm = confusion_matrix(y_test_m, y_pred_mnb)

print(f"Accuracy : {accuracy}")
print(f"Precision: {precision}")
print(f"Recall   : {recall}")
print(f"F1 Score : {f1}")
print("\nConfusion Matrix:\n", cm)

Accuracy : 0.8917105263157895
Precision: 0.8914017340556674
Recall   : 0.8917105263157895
F1 Score : 0.8914688089217605

Confusion Matrix:
 [[1691   61   92   56]
 [  34 1840    8   18]
 [  77   19 1596  208]
 [  77   20  153 1650]]


In [ ]:
import pickle

# Define the filename for the saved model
model_filename = 'multinomial_naive_bayes_model.pkl'

# Save the trained model to a .pkl file
with open(model_filename, 'wb') as file:
    pickle.dump(mnb_sklearn, file)

print(f"Trained model saved to {model_filename}")

Trained model saved to multinomial_naive_bayes_model.pkl


The model has been saved as `multinomial_naive_bayes_model.pkl`. You can load it back into memory using `pickle.load()` whenever you need to use it for predictions.

In [ ]:
import pickle
import numpy as np

# Load the trained model from the .pkl file
model_filename = 'multinomial_naive_bayes_model.pkl'
with open(model_filename, 'rb') as file:
    loaded_mnb_model = pickle.load(file)

print(f"Model loaded from {model_filename} successfully.")

Model loaded from multinomial_naive_bayes_model.pkl successfully.


Now, let's define a sample sentence, preprocess it, vectorize it using the existing `bow` object, and make a prediction.

In [ ]:
# Sample test sentence
sample_sentence = "The Zalmi storm will win the PSL"

print(f"Original sentence: {sample_sentence}")

# Preprocess the sample sentence
cleaned_sample_sentence = preprocess_text(sample_sentence)
print(f"Cleaned sentence: {cleaned_sample_sentence}")

# Vectorize the preprocessed sentence using the existing BoW model
vectorized_sample_sentence_dict = bow.vectorize(cleaned_sample_sentence)
print(f"Vectorized sentence (dictionary format): {vectorized_sample_sentence_dict}")

# Convert the dictionary to an array for prediction
# The vocab_size is already defined from previous steps
vectorized_sample_sentence_array = dicts_to_array([vectorized_sample_sentence_dict], vocab_size, dtype=np.uint8)
print(f"Vectorized sentence (array format shape): {vectorized_sample_sentence_array.shape}")

# Make a prediction using the loaded model
predicted_class = loaded_mnb_model.predict(vectorized_sample_sentence_array)

# The Class Index values are 1, 2, 3, 4. Let's map them to their categories for better understanding.
# 1: World, 2: Sports, 3: Business, 4: Sci/Tech
class_mapping = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
predicted_category = class_mapping.get(predicted_class[0], 'Unknown')

print(f"Predicted class index: {predicted_class[0]}")
print(f"Predicted category: {predicted_category}")

Original sentence: The Zalmi storm will win the PSL
Cleaned sentence: zalmi storm win psl
Vectorized sentence (dictionary format): {960: 1, 84: 1}
Vectorized sentence (array format shape): (1, 60525)
Predicted class index: 2
Predicted category: Sports


# Comparison of Manual and Sklearn implementation

The performance of the manual (from-scratch) models and the sklearn implementations is identical for both Bernoulli and Multinomial Naive Bayes. All evaluation metrics: Accuracy, Precision, Recall,
F1-score and the Confusion Matrices match exactly. This indicates that the manual implementations correctly replicate sklearn’s probability calculations and smoothing behavior. Any typical advantages of sklearn, such as better numerical stability or optimization, do not appear here because the dataset and implementation choices lead to the same predictions in both cases.